# NLP feature engineering — heuristic keyword scores

Sprint 5. A fast, fully interpretable NLP baseline before sentence
embeddings (Sprint 6). For each `review` body we count curated
flavour/aroma descriptor keywords and turn them into numeric features:

- `kw_<feature>_count` — raw keyword hits in the review
- `kw_<feature>`       — hits per 100 words (density, length-normalised)

This is the **only** place keyword scoring lives. Output is saved keyed
by `wine_id` as `wine_keywords.parquet`; the model notebook loads it and
joins on `wine_id` rather than recomputing.

In [ ]:
import re
import numpy as np
import pandas as pd
import itables
from itables import show

itables.options.columnDefs = [{"className": "dt-left", "targets": "_all"}]

SILVER_PATH   = r"..\..\.data\wine_reviews_silver.parquet"
KEYWORDS_PATH = r"..\..\.data\wine_keywords.parquet"

df = pd.read_parquet(SILVER_PATH)
print(f"Silver shape: {df.shape}")
print(f"reviews missing: {df['review'].isna().sum():,}")

## Keyword dictionaries

Curated descriptor lists per flavour/aroma axis. Edit freely — the rest
of the notebook is driven entirely by this dict.

In [ ]:
FEATURE_KEYWORDS = {
    "fruity": ["fruit", "fruity", "berry", "berries", "cherry", "cherries",
        "plum", "apple", "peach", "apricot", "citrus", "lemon", "lime",
        "orange", "grapefruit", "blackberry", "raspberry", "strawberry",
        "blueberry", "currant", "cassis", "melon", "pear", "fig", "tropical",
        "pineapple", "mango", "cranberry", "pomegranate"],
    "tannic": ["tannin", "tannins", "tannic", "grippy", "grip", "astringent",
        "firm", "chewy", "structured", "structure"],
    "acidic": ["acid", "acidity", "acidic", "crisp", "bright", "zesty", "zest",
        "tart", "fresh", "freshness", "lively", "vibrant", "racy", "tangy"],
    "oaky": ["oak", "oaky", "oaked", "vanilla", "cedar", "toast", "toasty",
        "smoky", "smoke", "barrel", "clove", "cinnamon", "spice", "spicy",
        "mocha", "espresso", "chocolate"],
    "sweet": ["sweet", "sweetness", "honey", "honeyed", "sugar", "sugary",
        "dessert", "candied", "jammy", "jam", "ripe", "luscious", "syrupy"],
    "body": ["full-bodied", "light-bodied", "medium-bodied", "rich", "heavy",
        "weighty", "concentrated", "dense", "powerful", "robust", "opulent",
        "lush", "plush"],
    "earthy": ["earth", "earthy", "mineral", "minerality", "leather",
        "leathery", "mushroom", "forest", "tobacco", "smoke", "flint", "stony"],
    "floral": ["floral", "flower", "rose", "violet", "blossom", "lavender",
        "jasmine", "honeysuckle", "perfumed"],
}

print(f"{len(FEATURE_KEYWORDS)} features, "
      f"{sum(len(v) for v in FEATURE_KEYWORDS.values())} keywords total")

## Compile patterns and score

One case-insensitive, word-boundaried alternation regex per feature.
`\\b` keeps hyphenated terms like `full-bodied` intact and stops
`oak` from matching `croak`.

In [ ]:
def build_patterns(keyword_dict):
    patterns = {}
    for feature, words in keyword_dict.items():
        alt = "|".join(re.escape(w) for w in sorted(words, key=len, reverse=True))
        patterns[feature] = re.compile(rf"\b(?:{alt})\b", re.IGNORECASE)
    return patterns


WORD_RE = re.compile(r"\b\w+\b")
PATTERNS = build_patterns(FEATURE_KEYWORDS)


def score_review(text):
    text = text or ""
    n_words = max(len(WORD_RE.findall(text)), 1)
    out = {}
    for feature, pat in PATTERNS.items():
        hits = len(pat.findall(text))
        out[f"kw_{feature}_count"] = hits
        out[f"kw_{feature}"] = round(100 * hits / n_words, 3)
    return out


reviews = df["review"].fillna("").astype(str)
kw_df = pd.DataFrame.from_records([score_review(t) for t in reviews], index=df.index)

count_cols   = [c for c in kw_df.columns if c.endswith("_count")]
density_cols = [c for c in kw_df.columns if not c.endswith("_count")]
print(f"Scored {len(kw_df):,} reviews -> {len(count_cols)} count + {len(density_cols)} density cols")
kw_df[density_cols].head()

## Coverage and distribution

Share of reviews mentioning each axis at least once, plus density stats.

In [ ]:
coverage = (kw_df[count_cols] > 0).mean().rename("pct_reviews_with_hit").mul(100).round(1)
avg_hits = kw_df[count_cols].mean().rename("avg_count").round(2)
summary = pd.concat([coverage, avg_hits], axis=1)
summary.index = summary.index.str.replace("kw_", "").str.replace("_count", "")
summary = summary.sort_values("pct_reviews_with_hit", ascending=False)
show(summary.reset_index().rename(columns={"index": "feature"}))

In [ ]:
kw_df[density_cols].describe().T.round(3)

## Spot-check

Sample a few reviews against their scores. Manually confirm the keyword
counts reflect what the text actually says.

In [ ]:
sample_idx = df.sample(8, random_state=7).index
show(
    pd.concat([df.loc[sample_idx, ["name", "review"]], kw_df.loc[sample_idx, count_cols]], axis=1)
    .reset_index(drop=True),
    maxBytes="2MB",
)

## Sanity check vs. price & rating

Quick correlation of the density features against the target (`retail`)
and `rating` to confirm they carry (weak) signal.

In [ ]:
signal = (
    pd.concat([kw_df[density_cols], df[["retail", "rating"]]], axis=1)
    .corr()[["retail", "rating"]]
    .drop(index=["retail", "rating"])
    .round(3)
    .sort_values("retail", ascending=False)
)
signal.index = signal.index.str.replace("kw_", "")
show(signal.reset_index().rename(columns={"index": "feature"}))

## Save keyword features

Persist `wine_id` + all `kw_*` columns. The model notebook joins these on
`wine_id` (NOT `slug`, which is not unique).

In [ ]:
out = pd.concat([df[["wine_id"]], kw_df], axis=1)
out.to_parquet(KEYWORDS_PATH, index=False)
print(f"Saved {out.shape[0]:,} rows x {out.shape[1]} cols -> {KEYWORDS_PATH}")